# 基于 PDE 的肿瘤药物渗透模拟
## 第二阶段：完整 FDM 求解器 + 动态可视化

**PDE 模型：**
$$\frac{\partial C}{\partial t} = D \nabla^2 C - \lambda C - k \cdot \rho(x,y) \cdot C$$

- $C(x,y,t)$：药物浓度场
- $D$：扩散系数
- $\lambda$：药物自然降解率
- $k$：细胞摄取率
- $\rho(x,y)$：细胞密度场（肿瘤核心密度高）

In [ ]:
# ============================================================
# 安装依赖
# ============================================================
!pip install plotly -q

In [ ]:
# ============================================================
# Phase 2: 完整 TumorModel
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')


class TumorModel:
    """
    2D 反应-扩散方程模拟肿瘤药物渗透。
    数值方法：显式有限差分法（FTCS）
    稳定性条件：dt <= dx^2 / (4D)
    """

    def __init__(self, grid_size=100, D=0.1, lam=0.01, k=0.05,
                 tumor_radius=25, tumor_center=None, drug_name='Generic Drug'):
        """
        参数说明：
          grid_size   : 网格尺寸（像素），代表 1cm × 1cm 区域
          D           : 扩散系数（无量纲，对应 ~1e-7 cm²/s 量级）
          lam         : 降解率
          k           : 细胞摄取率
          tumor_radius: 肿瘤半径（格点数）
          tumor_center: 肿瘤中心坐标，默认网格中心
          drug_name   : 药物名称（用于标注）
        """
        self.size = grid_size
        self.D = D
        self.lam = lam
        self.k = k
        self.drug_name = drug_name
        self.dx = 1.0 / grid_size
        self.dt = 0.9 * self.dx**2 / (4 * D + 1e-10)  # CFL 稳定性条件
        self.t = 0.0
        self.history = []  # 存储每一步快照

        # 初始化浓度场（全为 0）
        self.concentration = np.zeros((grid_size, grid_size))

        # 构建细胞密度场 rho(x,y)
        #   肿瘤核心：密度 1.0（高摄取）
        #   过渡区：  密度 0.3~0.8（梯度）
        #   正常组织：密度 0.1
        if tumor_center is None:
            cx, cy = grid_size // 2, grid_size // 2
        else:
            cx, cy = tumor_center

        self.cx, self.cy = cx, cy
        self.tumor_radius = tumor_radius
        self.rho = self._build_density_field(cx, cy, tumor_radius)

        # 扩散系数场：肿瘤核心扩散受阻（D 降低）
        self.D_field = self._build_diffusion_field(cx, cy, tumor_radius)

    def _build_density_field(self, cx, cy, r):
        """构建径向细胞密度场（肿瘤中心密度最高）"""
        xx, yy = np.meshgrid(np.arange(self.size), np.arange(self.size))
        dist = np.sqrt((xx - cx)**2 + (yy - cy)**2)
        rho = np.where(dist <= r,
                       1.0 - 0.5 * (dist / r),          # 肿瘤内部：1.0~0.5
                       0.1 + 0.4 * np.exp(-(dist - r) / (r * 0.3)))  # 外部快速衰减
        return np.clip(rho, 0.05, 1.0)

    def _build_diffusion_field(self, cx, cy, r):
        """构建空间变化扩散系数（肿瘤核心 D 降低 60%）"""
        xx, yy = np.meshgrid(np.arange(self.size), np.arange(self.size))
        dist = np.sqrt((xx - cx)**2 + (yy - cy)**2)
        D_field = np.where(dist <= r,
                           self.D * (0.4 + 0.6 * dist / r),  # 核心扩散受阻
                           self.D)
        return D_field

    def set_source(self, mode='left_boundary', concentration=1.0):
        """
        设置给药方式：
          'left_boundary' : 左侧边界持续给药（IV 输注模拟）
          'point'         : 点源给药（瘤内注射）
          'ring'          : 环形给药（血管包围）
        """
        self.source_mode = mode
        self.source_conc = concentration
        if mode == 'left_boundary':
            self.concentration[:, 0] = concentration
        elif mode == 'point':
            self.concentration[self.cy, self.cx] = concentration
        elif mode == 'ring':
            r = self.tumor_radius + 5
            xx, yy = np.meshgrid(np.arange(self.size), np.arange(self.size))
            dist = np.sqrt((xx - self.cx)**2 + (yy - self.cy)**2)
            self.concentration[np.abs(dist - r) < 2] = concentration

    def _apply_boundary(self):
        """维持源边界条件（持续给药）"""
        if self.source_mode == 'left_boundary':
            self.concentration[:, 0] = self.source_conc
            self.concentration[:, -1] = 0  # 右边界吸收
            self.concentration[0, :] = 0   # 上下边界
            self.concentration[-1, :] = 0

    def run_step(self, D=None, k=None, n_steps=1):
        """
        =====================================================
        核心 FDM 求解器：显式 FTCS 格式
        =====================================================
        离散化 PDE：
          C[i,j]^{n+1} = C[i,j]^n
            + dt * D[i,j] * Laplacian(C)[i,j]
            - dt * lambda * C[i,j]
            - dt * k * rho[i,j] * C[i,j]

        Laplacian（5点模板）：
          ∇²C ≈ (C[i+1,j] + C[i-1,j] + C[i,j+1] + C[i,j-1] - 4*C[i,j]) / dx²
        """
        D_use = D if D is not None else self.D
        k_use = k if k is not None else self.k

        for _ in range(n_steps):
            C = self.concentration
            # 计算 Laplacian（用 numpy roll 向量化，无 Python 循环）
            laplacian = (
                np.roll(C, 1, axis=0) +   # C[i-1, j]
                np.roll(C, -1, axis=0) +  # C[i+1, j]
                np.roll(C, 1, axis=1) +   # C[i, j-1]
                np.roll(C, -1, axis=1) -  # C[i, j+1]
                4 * C
            ) / (self.dx ** 2)

            # 更新浓度
            self.concentration = (
                C
                + self.dt * self.D_field * laplacian      # 扩散项
                - self.dt * self.lam * C                  # 降解项
                - self.dt * k_use * self.rho * C          # 摄取项
            )

            # 确保浓度非负
            self.concentration = np.clip(self.concentration, 0, None)

            # 维持边界条件
            self._apply_boundary()
            self.t += self.dt

    def get_penetration_curve(self):
        """
        计算径向渗透曲线：
        以肿瘤中心为原点，计算每个径向距离上的平均浓度
        """
        xx, yy = np.meshgrid(np.arange(self.size), np.arange(self.size))
        dist = np.sqrt((xx - self.cx)**2 + (yy - self.cy)**2).astype(int)
        max_r = int(np.sqrt(2) * self.size / 2)
        radial_conc = []
        radial_r = []
        for r in range(0, min(max_r, self.size // 2)):
            mask = dist == r
            if mask.sum() > 0:
                radial_conc.append(self.concentration[mask].mean())
                radial_r.append(r)
        return np.array(radial_r), np.array(radial_conc)

    def save_snapshot(self):
        """保存当前状态快照（用于动画）"""
        self.history.append((self.t, self.concentration.copy()))

    def display(self, ax=None, title=None):
        """matplotlib 热力图显示（兼容原始接口）"""
        if ax is None:
            fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.imshow(self.concentration, cmap='magma',
                       vmin=0, vmax=1, origin='lower')
        plt.colorbar(im, ax=ax, label='Concentration')
        # 绘制肿瘤边界
        circle = plt.Circle((self.cx, self.cy), self.tumor_radius,
                             color='cyan', fill=False, linewidth=1.5, linestyle='--')
        ax.add_patch(circle)
        ax.set_title(title or f'{self.drug_name} | t={self.t:.4f}')
        ax.set_xlabel('x (grid)')
        ax.set_ylabel('y (grid)')
        plt.tight_layout()


print('✅ TumorModel Phase 2 加载成功！')
print(f'   - 有限差分 FDM 求解器: ∂C/∂t = D∇²C - λC - k·ρ·C')
print(f'   - 空间变化扩散系数 D(x,y)')
print(f'   - 径向渗透曲线计算')
print(f'   - 三种给药模式: left_boundary / point / ring')

In [ ]:
# ============================================================
# 运行模拟 + 多时间步快照
# ============================================================

# 初始化模型（默认参数，可自由调整）
model = TumorModel(
    grid_size=100,
    D=0.08,           # 扩散系数
    lam=0.005,        # 降解率
    k=0.06,           # 摄取率
    tumor_radius=22,  # 肿瘤半径
    drug_name='Doxorubicin'
)
model.set_source(mode='left_boundary', concentration=1.0)

# 记录 5 个时间点的快照
snapshots = []
snapshot_steps = [0, 50, 150, 300, 500]

print('🔄 正在运行 PDE 模拟...')
step = 0
for target in snapshot_steps:
    steps_to_run = target - step
    if steps_to_run > 0:
        model.run_step(n_steps=steps_to_run)
    step = target
    snapshots.append((target, model.t, model.concentration.copy()))
    print(f'  ✓ Step {target:4d} | t={model.t:.5f} | max(C)={model.concentration.max():.4f}')

print('\n✅ 模拟完成！')

In [ ]:
# ============================================================
# 图 1：5 时间步热力图对比
# ============================================================
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle(f'{model.drug_name} Penetration into Tumor — Time Evolution', fontsize=14, y=1.02)

for idx, (s, t_val, C) in enumerate(snapshots):
    ax = axes[idx]
    im = ax.imshow(C, cmap='magma', vmin=0, vmax=1, origin='lower')
    circle = plt.Circle((model.cx, model.cy), model.tumor_radius,
                         color='cyan', fill=False, linewidth=1.5, linestyle='--')
    ax.add_patch(circle)
    ax.set_title(f'Step={s}\nt={t_val:.4f}', fontsize=9)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('drug_penetration_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 热力图已保存为 drug_penetration_heatmaps.png')

In [ ]:
# ============================================================
# 图 2：径向渗透曲线（Penetration Curves）
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(snapshots)))

# --- 左图：各时间步渗透曲线 ---
model2 = TumorModel(grid_size=100, D=0.08, lam=0.005, k=0.06,
                    tumor_radius=22, drug_name='Doxorubicin')
model2.set_source(mode='left_boundary', concentration=1.0)

for idx, (s, _, _) in enumerate(snapshots):
    # 重新运行到目标步
    m = TumorModel(grid_size=100, D=0.08, lam=0.005, k=0.06,
                   tumor_radius=22, drug_name='Doxorubicin')
    m.set_source(mode='left_boundary', concentration=1.0)
    if s > 0:
        m.run_step(n_steps=s)
    r, c = m.get_penetration_curve()
    ax1.plot(r, c, color=colors[idx], linewidth=2, label=f'Step={s}')

ax1.axvline(x=22, color='cyan', linestyle='--', alpha=0.7, label='Tumor boundary')
ax1.set_xlabel('Radial distance from tumor center (grid units)', fontsize=11)
ax1.set_ylabel('Mean drug concentration', fontsize=11)
ax1.set_title('Radial Penetration Curves', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
ax1.set_xlim(0, 50)

# --- 右图：最终步热力图 ---
_, _, C_final = snapshots[-1]
im = ax2.imshow(C_final, cmap='magma', vmin=0, vmax=1, origin='lower')
circle = plt.Circle((model.cx, model.cy), model.tumor_radius,
                     color='cyan', fill=False, linewidth=2, linestyle='--', label='Tumor')
ax2.add_patch(circle)
plt.colorbar(im, ax=ax2, label='Drug Concentration')
ax2.set_title(f'Final Concentration Field (Step={snapshot_steps[-1]})', fontsize=12)
ax2.set_xlabel('x (grid)')
ax2.set_ylabel('y (grid)')

plt.suptitle(f'{model.drug_name} — PDE Simulation Results\nD={model.D}, λ={model.lam}, k={model.k}',
             fontsize=13, y=1.03)
plt.tight_layout()
plt.savefig('penetration_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('📈 渗透曲线已保存为 penetration_curves.png')

In [ ]:
# ============================================================
# 图 3：参数扫描 — 不同扩散系数 D 对比
# ============================================================
D_values = [0.02, 0.05, 0.10, 0.20]
colors2 = plt.cm.viridis(np.linspace(0.1, 0.9, len(D_values)))

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for idx, D_val in enumerate(D_values):
    m = TumorModel(grid_size=100, D=D_val, lam=0.005, k=0.06, tumor_radius=22)
    m.set_source(mode='left_boundary', concentration=1.0)
    m.run_step(n_steps=300)

    # 热力图
    ax_heat = axes[0, idx]
    im = ax_heat.imshow(m.concentration, cmap='magma', vmin=0, vmax=1, origin='lower')
    circle = plt.Circle((m.cx, m.cy), m.tumor_radius,
                         color='cyan', fill=False, linewidth=1.5, linestyle='--')
    ax_heat.add_patch(circle)
    ax_heat.set_title(f'D = {D_val}', fontsize=11)
    ax_heat.axis('off')
    plt.colorbar(im, ax=ax_heat, fraction=0.046)

    # 渗透曲线
    ax_curve = axes[1, idx]
    r, c = m.get_penetration_curve()
    ax_curve.plot(r, c, color=colors2[idx], linewidth=2.5)
    ax_curve.axvline(x=22, color='cyan', linestyle='--', alpha=0.6)
    ax_curve.set_xlabel('Radial distance', fontsize=9)
    ax_curve.set_ylabel('Concentration', fontsize=9)
    ax_curve.set_xlim(0, 50)
    ax_curve.set_ylim(0, 1)
    ax_curve.grid(alpha=0.3)

    # 计算 50% 渗透深度
    half_max = c.max() / 2 if c.max() > 0 else 0
    penetration_depth = r[c >= half_max].max() if (c >= half_max).any() else 0
    ax_curve.set_title(f'50% depth={penetration_depth:.0f}px', fontsize=9)

fig.suptitle('Parameter Sweep: Effect of Diffusion Coefficient D (Step=300)', fontsize=14)
plt.tight_layout()
plt.savefig('parameter_sweep_D.png', dpi=150, bbox_inches='tight')
plt.show()
print('🔬 参数扫描结果已保存为 parameter_sweep_D.png')

In [ ]:
# ============================================================
# 图 4：Plotly 交互式热力图（可缩放/悬停）
# ============================================================

# 重新运行一次得到最终结果
m_final = TumorModel(grid_size=100, D=0.08, lam=0.005, k=0.06,
                     tumor_radius=22, drug_name='Doxorubicin')
m_final.set_source(mode='left_boundary', concentration=1.0)
m_final.run_step(n_steps=500)
r_final, c_final = m_final.get_penetration_curve()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Drug Concentration Heatmap (Interactive)', 'Radial Penetration Curve'),
    column_widths=[0.55, 0.45]
)

# 交互式热力图
fig.add_trace(
    go.Heatmap(
        z=m_final.concentration,
        colorscale='Magma',
        zmin=0, zmax=1,
        colorbar=dict(title='Concentration', x=0.52),
        hovertemplate='x=%{x}<br>y=%{y}<br>C=%{z:.4f}<extra></extra>'
    ),
    row=1, col=1
)

# 肿瘤边界圆
theta = np.linspace(0, 2 * np.pi, 200)
fig.add_trace(
    go.Scatter(
        x=m_final.cx + m_final.tumor_radius * np.cos(theta),
        y=m_final.cy + m_final.tumor_radius * np.sin(theta),
        mode='lines',
        line=dict(color='cyan', width=2, dash='dash'),
        name='Tumor boundary',
        showlegend=True
    ),
    row=1, col=1
)

# 渗透曲线
fig.add_trace(
    go.Scatter(
        x=r_final, y=c_final,
        mode='lines',
        line=dict(color='#FF6B6B', width=3),
        name='Radial concentration',
        hovertemplate='r=%{x}<br>C=%{y:.4f}<extra></extra>'
    ),
    row=1, col=2
)

# 肿瘤边界线
fig.add_vline(x=22, line_dash='dash', line_color='cyan',
              annotation_text='Tumor wall', row=1, col=2)

fig.update_layout(
    title=dict(
        text=f'<b>{m_final.drug_name}</b> Penetration Simulation<br>'
             f'<span style="font-size:12px">D={m_final.D} | λ={m_final.lam} | k={m_final.k} | Step=500</span>',
        x=0.5
    ),
    height=480,
    paper_bgcolor='#0d1117',
    plot_bgcolor='#0d1117',
    font=dict(color='white'),
    legend=dict(x=0.55, y=0.98)
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.write_html('interactive_simulation.html')
fig.show()
print('🌐 交互式图表已保存为 interactive_simulation.html')

In [ ]:
# ============================================================
# 图 5：三种给药方式对比
# ============================================================
modes = [
    ('left_boundary', 'IV Infusion (Left Boundary)'),
    ('point',         'Intratumoral Injection (Point)'),
    ('ring',          'Vascular Ring Delivery'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for idx, (mode, label) in enumerate(modes):
    m = TumorModel(grid_size=100, D=0.08, lam=0.005, k=0.06, tumor_radius=22)
    m.set_source(mode=mode, concentration=1.0)
    m.run_step(n_steps=400)

    ax_h = axes[0, idx]
    im = ax_h.imshow(m.concentration, cmap='inferno', vmin=0, vmax=1, origin='lower')
    circle = plt.Circle((m.cx, m.cy), m.tumor_radius,
                         color='lime', fill=False, linewidth=2, linestyle='--')
    ax_h.add_patch(circle)
    ax_h.set_title(label, fontsize=10, fontweight='bold')
    ax_h.axis('off')
    plt.colorbar(im, ax=ax_h, fraction=0.046)

    ax_c = axes[1, idx]
    r, c = m.get_penetration_curve()
    ax_c.fill_between(r, c, alpha=0.3, color='orangered')
    ax_c.plot(r, c, color='orangered', linewidth=2)
    ax_c.axvline(x=22, color='lime', linestyle='--', alpha=0.7, label='Tumor wall')
    ax_c.set_xlim(0, 50)
    ax_c.set_ylim(0, 1)
    ax_c.set_xlabel('Radial distance (px)')
    ax_c.set_ylabel('Concentration')
    ax_c.grid(alpha=0.3)

    tumor_mean = m.concentration[
        np.sqrt((np.arange(100)[:, None] - m.cx)**2 +
                (np.arange(100)[None, :] - m.cy)**2) <= 22
    ].mean()
    ax_c.set_title(f'Tumor avg C = {tumor_mean:.3f}', fontsize=9)

fig.suptitle('Delivery Mode Comparison (Step=400)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('delivery_modes_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('💊 给药方式对比已保存为 delivery_modes_comparison.png')

In [ ]:
# ============================================================
# 汇总报告输出
# ============================================================
print('=' * 60)
print('        PDE 肿瘤药物渗透模拟 — 第二阶段完成')
print('=' * 60)
print()
print('✅ 已实现功能：')
print('   1. 完整 FDM 求解器（∂C/∂t = D∇²C - λC - k·ρ·C）')
print('   2. 空间变化扩散系数 D(x,y) + 细胞密度场 ρ(x,y)')
print('   3. 时间演化热力图（5个时间步）')
print('   4. 径向渗透曲线分析')
print('   5. 参数扫描（D 系数对比）')
print('   6. Plotly 交互式可视化 → interactive_simulation.html')
print('   7. 三种给药方式对比')
print()
print('📁 生成文件：')
print('   - drug_penetration_heatmaps.png')
print('   - penetration_curves.png')
print('   - parameter_sweep_D.png')
print('   - interactive_simulation.html  ← 可直接嵌入网站')
print('   - delivery_modes_comparison.png')
print()
print('🔜 下一步（Phase 3）：')
print('   - 构建 Streamlit / Gradio 前端 UI')
print('   - 接入 Claude API：用户输入药物名 → 自动获取参数')
print('   - 部署到 Hugging Face Spaces')